# Smogzilla scrapers — the idea

There are **2 scrapers**, both subclasses of `BaseScraper`:

| Class | Where it pulls from | What kind of station |
|---|---|---|
| `SensorCommunityScraper` | sensor.community | citizen-hosted cheap sensors |
| `WaqiScraper` | api.waqi.info (aqicn.org) | official government stations (ARPA in Italy) |

Both implement the same method:
```python
await scraper.fetch_by_area(lat, lng, radius_km) -> list[SensorReading]
```
Each reading is tagged with `"source"` so we can merge them later.

Both scrapers drop readings older than **3 hours** (`scrapers.types.MAX_AGE_HOURS`) — official stations can go silent without warning, and we don't want stale data in reports.

**Setup**: from the project root run `pip install -e .` once. After that, all imports just work — no `sys.path` hacks needed.


## WAQI token

Already configured in `.env` as `WAQI_API_TOKEN`. Loaded automatically by `config.py`.

If you ever need to regenerate: https://aqicn.org/data-platform/token/

## Pick a city

All supported cities live in `utils.city_data.CITIES`.

In [1]:
from utils.city_data import CITIES

print(f"{len(CITIES)} cities supported")
print("Sample:", list(CITIES)[:8])

city = CITIES["milano"]
print(f"\nUsing: {city.name} ({city.lat}, {city.lng})")

48 cities supported
Sample: ['torino', 'milano', 'brescia', 'bergamo', 'monza', 'como', 'varese', 'pavia']

Using: Milano (45.4654, 9.1859)


## Live test

Calls both scrapers against their real APIs and prints side-by-side. Requires internet.

In [2]:
from config import WAQI_API_TOKEN
from scrapers.sensor_community import SensorCommunityScraper
from scrapers.waqi import WaqiScraper
from datetime import datetime, timezone

community = await SensorCommunityScraper().fetch_by_area(city.lat, city.lng)
waqi      = await WaqiScraper(token=WAQI_API_TOKEN).fetch_by_area(city.lat, city.lng)

def age(ts):
    # sensor.community is naive (assumed UTC), WAQI carries tz info
    t = datetime.fromisoformat(ts.replace(" ", "T"))
    if t.tzinfo is None:
        t = t.replace(tzinfo=timezone.utc)
    return (datetime.now(timezone.utc) - t).total_seconds() / 60

for r in community + waqi:
    a = age(r["timestamp"])
    print(f"[{r['source']:>16}]  sensor {r['sensor_id']:>8}  "
          f"pm2.5={r['pm2.5']!r:>7}  pm10={r['pm10']!r:>7}  ({a:.0f} min ago)")
print(f"\n{len(community)} community + {len(waqi)} waqi readings for {city.name}")
print(f"(readings older than {__import__('scrapers.types', fromlist=['MAX_AGE_HOURS']).MAX_AGE_HOURS}h are filtered out)")

[sensor.community]  sensor    32393  pm2.5=    7.0  pm10=   9.25  (0 min ago)
[sensor.community]  sensor    21665  pm2.5=   3.15  pm10=   4.45  (1 min ago)
[sensor.community]  sensor    57418  pm2.5=   3.78  pm10=   6.97  (1 min ago)
[sensor.community]  sensor    32399  pm2.5=   6.97  pm10=  10.52  (1 min ago)
[sensor.community]  sensor    99130  pm2.5=   23.0  pm10=   23.0  (1 min ago)
[sensor.community]  sensor    98873  pm2.5=11.3700428009033  pm10=11.477972984314  (2 min ago)
[sensor.community]  sensor    98875  pm2.5=4.84558391571045  pm10=4.84558582305908  (2 min ago)
[sensor.community]  sensor    98877  pm2.5=10.1569328308105  pm10=10.1569318771362  (2 min ago)
[sensor.community]  sensor    60342  pm2.5=   6.07  pm10=    9.2  (2 min ago)
[sensor.community]  sensor    59970  pm2.5=   1.73  pm10=    2.7  (2 min ago)
[sensor.community]  sensor    40659  pm2.5=    5.1  pm10=    9.6  (2 min ago)
[sensor.community]  sensor    98890  pm2.5=   9.89  pm10=   9.89  (3 min ago)
[sensor.com

## One thing to notice about WAQI

WAQI returns **IAQI** (0–500 scale), not µg/m³. The scraper converts via US-EPA piecewise breakpoints.

| IAQI | PM2.5 µg/m³ |
|---|---|
| 50   | 12.0 |
| 100  | 35.4 |
| 150  | 55.4 |
| 200  | 150.4 |
